# **Packages**

In [ ]:
!pip install opencv-python==4.12.0
!pip install transformers==4.56.0
!pip install python-dotenv
!pip install torch
!pip install scipy

# **HF Auth vs setting device**

In [ ]:
import os
import torch
from pathlib import Path
from huggingface_hub import login, HfApi
from dotenv import load_dotenv

load_dotenv()

token = os.environ.get("HF_TOKEN")
login(token=token, add_to_git_credential=True)

api = HfApi()
user_info = api.whoami()
print(f"✅ Đã đăng nhập Hugging Face: {user_info.get('name') or user_info.get('login')}")

# Device selection
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print("cuda available:", torch.cuda.is_available(),
      "| device count:", torch.cuda.device_count(),
      "| current device:", torch.cuda.current_device())
    gpu_stats = torch.cuda.get_device_properties(device)
    max_memory = round(gpu_stats.total_memory / 1024**3, 3)
    print(f"   GPU: {gpu_stats.name}")
    print(f"   Dung lượng tối đa: {max_memory} GB")
    try:
        print("Device name:", torch.cuda.get_device_name(torch.cuda.current_device()))
    except Exception:
        pass
else:
    max_memory = 0

# **Utils**

In [2]:
from pathlib import Path
from typing import List, Optional

def resolve_path_candidates(relative_path: str) -> List[Path]:
    """Trả về các đường dẫn dự phòng để thử:
    1. current_path.parents[1] / relative_path
    2. current_path / relative_path
    """
    current_path = Path().resolve()
    candidates = []
    try:
        candidates.append((current_path.parents[1] / relative_path).resolve())
    except IndexError:
        pass
    candidates.append((current_path / relative_path).resolve())
    return candidates


def find_existing_path(relative_path: str) -> Optional[Path]:
    for candidate in resolve_path_candidates(relative_path):
        if candidate.exists():
            print(f"✅ Found: {candidate}")
            return candidate
        print(f"⚠️ Not found: {candidate}")
    print(f"❌ None of the candidates for '{relative_path}' exist.")
    return None

# **Initilize Architecture**

In [1]:
import math
import torch
import torch.nn as nn
from transformers import AutoModel

class PositionEmbeddingSine(nn.Module):
  def __init__(
      self,
      number_position_features=128, # if hidden_dim=256 --> divide half for x and y
      temperature=10000,
      normalize=True,
      scale=None
  ):
    super().__init__()
    self.number_position_features = number_position_features
    self.temperature = temperature
    self.normalize = normalize
    if scale is None:
      scale = 2*math.pi
    self.scale = scale

  def forward(self, image_mask):
    if image_mask is None:
      raise ValueError("PositionEmbeddingSine yêu cầu mask để biết kích thước ảnh.")
    else:
      valid_area_mask = ~image_mask
      y_coords = valid_area_mask.cumsum(1, dtype=torch.float32)
      x_coords = valid_area_mask.cumsum(2, dtype=torch.float32)

    if self.normalize:
      eps = 1e-6
      y_coords = y_coords / (y_coords[:, -1:, :] + eps) * self.scale
      x_coords = x_coords / (x_coords[:, :, -1:] + eps) * self.scale

    dimension_temperature_vector = torch.arange(self.number_position_features, dtype=torch.float32, device=image_mask.device)
    dimension_temperature_vector = self.temperature ** (2 * (dimension_temperature_vector // 2) / self.number_position_features)

    position_x_encode = x_coords[:, :, :, None] / dimension_temperature_vector
    position_y_encode = y_coords[:, :, :, None] / dimension_temperature_vector
    position_x_encode = torch.stack((position_x_encode[..., 0::2].sin(), position_x_encode[..., 1::2].cos()), dim=4).flatten(3)
    position_y_encode = torch.stack((position_y_encode[..., 0::2].sin(), position_y_encode[..., 1::2].cos()), dim=4).flatten(3)
    final_positional_embedding = torch.cat((position_y_encode, position_x_encode), dim=3).permute(0, 3, 1, 2) # [B, H, W, 2*number_position_features]
    return final_positional_embedding

class DETRHead(nn.Module):
  def __init__(
      self,
      backbone_dim,
      num_object_queries,
      num_classes,
      hidden_dim=256,
      num_attention_heads=8,
      num_encoder_layers=3,
      num_decoder_layers=3,
      feed_forward_dim=1024,
      dropout_rate=0.1
  ):
    super().__init__()
    self.hidden_dim = hidden_dim
    self.num_object_queries = num_object_queries
    self.num_classes = num_classes
    self.input_projection = nn.Conv2d(backbone_dim, hidden_dim, kernel_size=1)

    self.transformer = nn.Transformer(
        d_model=hidden_dim,
        nhead=num_attention_heads,
        num_encoder_layers=num_encoder_layers,
        num_decoder_layers=num_decoder_layers,
        dim_feedforward=feed_forward_dim,
        dropout=dropout_rate,
        activation='relu',
        batch_first=False # (S,N,E) AS REQUIRED
    )

    self.object_queries = nn.Embedding(num_object_queries, hidden_dim)
    self.class_predictor = nn.Linear(hidden_dim, num_classes + 1) # =1 due to no object at first
    self.bbox_predictor = nn.Linear(hidden_dim, 4)

    # positional encoding (with half of hidden dimension per axis)
    self.positional_encoder = PositionEmbeddingSine(number_position_features=hidden_dim // 2, normalize=True)

  def forward(self, backbone_features, image_mask=None):
    B, H, W, _ = backbone_features.shape
    device = backbone_features.device
    projected_features = self.input_projection(backbone_features.permute(0, 3, 1, 2))
    if image_mask is None:
      image_mask = torch.zeros((B, H, W), dtype=torch.bool, device=device)

    positional_embedding = self.positional_encoder(image_mask)

    # Pre-process input for Transformer: flatten dims -> seq
    encoder_input_sequence = projected_features.flatten(2).permute(2, 0, 1)
    positional_embedding_sequence = positional_embedding.flatten(2).permute(2, 0, 1)

    encoder_input_with_positional = encoder_input_sequence + positional_embedding_sequence

    # Object queries cho decoder
    object_query_sequence = self.object_queries.weight.unsqueeze(1).repeat(1, B, 1)
    padding_mask_sequence = image_mask.flatten(1)

    transformer_output = self.transformer(
            src=encoder_input_with_positional,
            tgt=object_query_sequence,
            src_key_padding_mask=padding_mask_sequence,
    )
    decoder_output_sequence = transformer_output.permute(1, 0, 2)
    # Heads, prediction
    predicted_logits = self.class_predictor(decoder_output_sequence)        # [B, T, num_classes+1]
    predicted_boxes = self.bbox_predictor(decoder_output_sequence).sigmoid()# [B, T, 4]

    return {
        'pred_logits': predicted_logits,
        'pred_boxes': predicted_boxes
    }

class DINODetector(nn.Module):
  def __init__(self, backbone_model, num_object_queries=15, num_classes=1):
    super().__init__()
    self.backbone = backbone_model
    backbone_output_dim = self.backbone.config.hidden_size
    self.detection_head = DETRHead(
        backbone_dim=backbone_output_dim,
        num_object_queries=num_object_queries,
        num_classes=num_classes
    )

  def forward(self, pixel_values, image_mask=None, return_features=False):
    # Lấy output từ backbone DINOv3
    backbone_outputs = self.backbone(pixel_values=pixel_values)
    all_tokens = backbone_outputs.last_hidden_state # Shape: [B, total_tokens, C]

    B, _, H_img, W_img = pixel_values.shape
    patch_size = self.backbone.config.patch_size
    assert H_img % patch_size == 0 and W_img % patch_size == 0, \
    f"Image H,W must be divisible by patch_size ({patch_size}). Got H={H_img}, W={W_img}."

    # Tính toán số lượng patch dự kiến
    h_patches = H_img // patch_size
    w_patches = W_img // patch_size
    num_patches = h_patches * w_patches
    num_special_tokens = all_tokens.shape[1] - num_patches

    # Lấy ra các patch embeddings bằng cách bỏ đi các token thừa ở đầu
    patch_embeddings = all_tokens[:, num_special_tokens:, :]

    # Reshape thành lưới 2D
    feature_grid_2d = patch_embeddings.reshape(B, h_patches, w_patches, -1)

    # Đưa vào decoder head để lấy dự đoán cuối cùng
    predictions = self.detection_head(feature_grid_2d, image_mask=image_mask)

    if return_features:
      return feature_grid_2d, predictions
    else:
      return predictions

# **Dataset vs DataLoader**

In [ ]:
import os
import glob
import math
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

def find_image_files(directory, extensions=("jpg", "jpeg", "png")):
    directory = Path(directory)
    if not directory.exists():
        return []
    image_paths = []
    for ext in extensions:
        image_paths.extend([str(p) for p in directory.rglob(f"*.{ext}")])
    return sorted(image_paths)


def parse_yolo_label_file(label_path):
  boxes = []
  labels = []
  p = Path(label_path)
  if not p.exists():
    return boxes, labels

  with p.open("r") as f:
    for line in f:
      parts = line.strip().split()
      if len(parts) < 5:
        continue
      try:
        class_id = int(parts[0])
        center_x, center_y, width, height = map(float, parts[1:5])
      except Exception:
        # skip malformed lines
        continue
      boxes.append([center_x, center_y, width, height])
      labels.append(class_id)
  return boxes, labels


class ObjectDetectionDataset(Dataset):
  def __init__(self, image_paths, processor, augmentations=None, debug=False):
      self.image_paths = image_paths
      self.processor = processor
      self.augmentations = augmentations
      self.debug = debug

  def __len__(self):
      return len(self.image_paths)

  def __getitem__(self, index):
    image_path = self.image_paths[index]
    image = Image.open(image_path).convert("RGB")
    original_width, original_height = image.size
    if self.augmentations is not None:
      image = self.augmentations(image)

    processor_output = self.processor(images=image, return_tensors="pt")
    pixel_values = processor_output["pixel_values"].squeeze(0) # Bỏ chiều batch [1,C,H,W] -> [C,H,W]

    label_path = Path(image_path).with_suffix('.txt')
    boxes, labels = parse_yolo_label_file(label_path)

    # convert to tensors
    if len(boxes) > 0:
      boxes_tensor = torch.as_tensor(boxes, dtype=torch.float32)
      labels_tensor = torch.as_tensor(labels, dtype=torch.int64)
    else:
      # handle images with no objects
      boxes_tensor = torch.zeros((0, 4), dtype=torch.float32)
      labels_tensor = torch.zeros((0,), dtype=torch.int64)

    target = {
      "boxes": boxes_tensor,
      "labels": labels_tensor,
      "image_id": Path(image_path).stem,
      "original_size": torch.tensor([original_height, original_width], dtype=torch.int32)
    }
    return pixel_values, target

def detection_collate_fn(batch):
    pixel_values_list = [item[0] for item in batch]
    targets_list = [item[1] for item in batch]
    pixel_values_batch = torch.stack(pixel_values_list, dim=0)
    return pixel_values_batch, targets_list

def create_augmentations(is_train=True):
    if is_train:
        return T.Compose([
            T.RandomHorizontalFlip(p=0.5),
            T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
        ])
    else:
        return None

# **Loss function**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.optimize import linear_sum_assignment

def convert_boxes_from_center_to_corners(boxes_center_format): # from (center_x, center_y, width, height) to (x_min, y_min, x_max, y_max).
  center_x, center_y, width, height = boxes_center_format.unbind(-1)
  x_min = center_x - 0.5 * width
  y_min = center_y - 0.5 * height
  x_max = center_x + 0.5 * width
  y_max = center_y + 0.5 * height
  return torch.stack([x_min, y_min, x_max, y_max], dim=-1) # torch.stack tạo tensor cuối cùng shape [...,4] từ các thành phần từng phần; giữ định dạng [x_min,y_min,x_max,y_max]

def calculate_box_area(boxes_corner_format):
  # boxes: [..., 4] in x0,y0,x1,y1
    x_min, y_min, x_max, y_max = boxes_corner_format.unbind(-1)
    return (x_max - x_min).clamp(min=0) * (y_max - y_min).clamp(min=0)

def calculate_generalized_box_iou(boxes1_corners, boxes2_corners): # Generalized Intersection over Union (GIoU), chỉ số nâng cao của IoU, cung cấp loss signal ngay cả khi 2 boxes không giao nhau
  # intersection
  num_boxes1, num_boxes2 = boxes1_corners.shape[0], boxes2_corners.shape[0]
  if num_boxes1 == 0 or num_boxes2 == 0:
    return torch.zeros((num_boxes1, num_boxes2), device=boxes1_corners.device)

  boxes1_expanded = boxes1_corners.unsqueeze(1) # boxes1: [N, 4] -> [N, 1, 4]
  boxes2_expanded = boxes2_corners.unsqueeze(0) # boxes2: [M, 4] -> [1, M, 4]

  # Vùng giao nhau
  intersection_top_left = torch.max(boxes1_expanded[..., :2], boxes2_expanded[..., :2])
  intersection_bottom_right = torch.min(boxes1_expanded[..., 2:], boxes2_expanded[..., 2:])
  intersection_wh = (intersection_bottom_right - intersection_top_left).clamp(min=0)
  intersection_area = intersection_wh[..., 0] * intersection_wh[..., 1] # Shape: [N, M]

  # Vùng Union
  area1 = calculate_box_area(boxes1_corners) # Shape: [N]
  area2 = calculate_box_area(boxes2_corners) # Shape: [M]
  union_area = area1.unsqueeze(1) + area2.unsqueeze(0) - intersection_area # Shape: [N, M]

  # IoU
  epsilon = 1e-7
  iou = intersection_area / (union_area + epsilon)

  # Smallest Enclosing Box Calculation --> For what?
  enclosing_top_left = torch.min(boxes1_expanded[..., :2], boxes2_expanded[..., :2])
  enclosing_bottom_right = torch.max(boxes1_expanded[..., 2:], boxes2_expanded[..., 2:])
  enclosing_wh = (enclosing_bottom_right - enclosing_top_left).clamp(min=0)
  enclosing_area = enclosing_wh[..., 0] * enclosing_wh[..., 1]

  # GIoU
  giou = iou - (enclosing_area - union_area) / (enclosing_area + epsilon)
  return giou

class HungarianMatcher(nn.Module):
  def __init__(self, cost_class: float = 1.0, cost_bbox_l1: float = 5.0, cost_giou: float = 2.0, verbose=False):
    super().__init__()
    self.cost_class = cost_class
    self.cost_bbox_l1 = cost_bbox_l1
    self.cost_giou = cost_giou
    self.verbose = verbose

  @torch.no_grad()
  def forward(self, model_outputs, ground_truth_targets):
    batch_size, num_queries = model_outputs['pred_logits'].shape[:2]

    # Lấy ra các dự đoán từ output của model
    pred_probabilities = model_outputs['pred_logits'].softmax(-1)
    pred_boxes_center = model_outputs['pred_boxes'] # Định dạng [cx, cy, w, h]

    if self.verbose:
      print(f"pred_logits.shape: {model_outputs['pred_logits'].shape}")
      print(f"pred_boxes.shape: {pred_boxes_center.shape}")
      print(f"batch_size: {batch_size}, num_queries: {num_queries}")
      print(f"num_classes(+bg): {model_outputs['pred_logits'].shape[-1]}")

      for i, t in enumerate(ground_truth_targets):
        boxes_shape = t['boxes'].shape if 'boxes' in t else None
        labels_shape = t['labels'].shape if 'labels' in t else None
        print(f" target[{i}] boxes: {boxes_shape}, labels: {labels_shape}")

    indices = []
    for i in range(batch_size):
        target_labels = ground_truth_targets[i]['labels']
        target_boxes_center = ground_truth_targets[i]['boxes']

        if target_labels.numel() == 0: # Xử lý ảnh không có đối tượng
            indices.append((
                torch.tensor([], dtype=torch.int64), torch.tensor([], dtype=torch.int64)
            ))
            if self.verbose:
              print(f"[Matcher] batch {i}: no targets (num_tgt=0).")
            continue

        # 1.Classification Cost
        cost_class = -pred_probabilities[i][:, target_labels] # Shape: [num_queries, num_targets]

        # 2.L1 BBox Cost
        cost_bbox_l1 = torch.cdist(pred_boxes_center[i], target_boxes_center, p=1)

        # 3. GIoU BBox Cost
        pred_boxes_corners = convert_boxes_from_center_to_corners(pred_boxes_center[i])
        target_boxes_corners = convert_boxes_from_center_to_corners(target_boxes_center)
        cost_giou = -calculate_generalized_box_iou(pred_boxes_corners, target_boxes_corners)

        # 4. Final Cost Matrix
        cost_matrix = (self.cost_bbox_l1 * cost_bbox_l1 +
                        self.cost_class * cost_class +
                        self.cost_giou * cost_giou)

        # 5.
        # Thuật toán tìm các cặp (dự đoán, target) sao cho tổng chi phí là nhỏ nhất.
        cost_matrix_cpu = cost_matrix.detach().cpu().numpy()
        row_indices, col_indices = linear_sum_assignment(cost_matrix_cpu)

        # Chuyển chỉ số về lại tensor
        row_indices = torch.as_tensor(row_indices, dtype=torch.int64)
        col_indices = torch.as_tensor(col_indices, dtype=torch.int64)
        indices.append((row_indices, col_indices))

    return [(r.to(pred_boxes_center.device), c.to(pred_boxes_center.device)) for r, c in indices]

class DETRLoss(nn.Module):
  def __init__(
      self,
      num_classes,
      matcher,
      weight_dict,
      no_object_class_weight,
      verbose=False
  ):
    super().__init__()
    self.num_classes = num_classes
    self.matcher = matcher
    self.weight_dict = weight_dict
    self.no_object_class_weight = no_object_class_weight
    self.verbose = verbose

    class_weights = torch.ones(self.num_classes + 1)
    class_weights[-1] = self.no_object_class_weight
    self.register_buffer('class_weights', class_weights)

    if self.verbose:
        print(f"num_classes: {self.num_classes}, no_object_class_weight: {self.no_object_class_weight}")
        print(f"class_weight shape: {self.class_weights.shape}")
        print(f"weight_dict: {self.weight_dict}")

  @torch.no_grad()
  def loss_cardinality(self, outputs, targets):
    # Count predicted non-empty (predicted class != no-object)
    pred_logits = outputs['pred_logits']
    device = pred_logits.device
    tgt_lengths = torch.as_tensor([len(v["labels"]) for  v in targets], device=device)
    # predicted counts = number of predictions whose argmax != last class
    card_pred = (pred_logits.argmax(-1) != self.num_classes).sum(1)
    card_err = F.l1_loss(card_pred.float(), tgt_lengths.float())
    return {'cardinality_error': card_err}

  def _get_permutation_indices(self, matched_indices):
    """Hàm trợ giúp để lấy ra chỉ số của các cặp đã được so khớp."""
    batch_indices = torch.cat([torch.full_like(src, i) for i, (src, _) in enumerate(matched_indices)])
    prediction_indices = torch.cat([src for (src, _) in matched_indices])

    target_indices = torch.cat([tgt for (_, tgt) in matched_indices])
    return (batch_indices, prediction_indices), (batch_indices, target_indices)

  def forward(self, model_outputs, ground_truth_targets):
    if self.verbose:
        print(f"outputs['pred_logits'].shape: {model_outputs['pred_logits'].shape}")
        print(f"outputs['pred_boxes'].shape: {model_outputs['pred_boxes'].shape}")
        print(f"num_classes (without bg): {self.num_classes}")
        print(f"class_weight: {self.class_weights.cpu().numpy()}")
    # Find match sets
    matched_indices = self.matcher(model_outputs, ground_truth_targets)
    # classification loss
    pred_logits = model_outputs['pred_logits']
    batch_size, num_queries, _ = pred_logits.shape
    # prepare target classes for each prediction (default = no-object)
    target_classes = torch.full((batch_size, num_queries), self.num_classes, dtype=torch.int64, device=pred_logits.device)

    # Gán class thật cho các dự đoán đã được so khớp
    for batch_idx, (pred_idx, target_idx) in enumerate(matched_indices):
      if pred_idx.numel() == 0:
          continue
      target_classes[batch_idx, pred_idx] = ground_truth_targets[batch_idx]['labels'][target_idx]

    # Cross-Entropy loss cho tất cả các queries
    loss_classification = F.cross_entropy(pred_logits.transpose(1, 2), target_classes, self.class_weights)

    # Loss Hồi Quy Bounding Box (chỉ cho các cặp đã so khớp)
    (batch_indices, pred_indices), (target_batch_indices, target_indices) = self._get_permutation_indices(matched_indices)

    if pred_indices.numel() > 0:
      matched_pred_boxes = model_outputs['pred_boxes'][batch_indices, pred_indices]
      matched_target_boxes = torch.cat([t['boxes'][i] for t, (_, i) in zip(ground_truth_targets, matched_indices)], dim=0)

      # loss L1
      loss_bbox_l1 = F.l1_loss(matched_pred_boxes, matched_target_boxes, reduction='mean')

      # Loss GIoU
      matched_pred_boxes_corners = convert_boxes_from_center_to_corners(matched_pred_boxes)
      matched_target_boxes_corners = convert_boxes_from_center_to_corners(matched_target_boxes)

      # GIoU được tính pairwise, ta chỉ cần lấy đường chéo chính
      giou_matrix = calculate_generalized_box_iou(matched_pred_boxes_corners, matched_target_boxes_corners)
      loss_giou = (1 - torch.diag(giou_matrix)).mean()
    else: # nếu ko có cặp nào khớp --> return 0
      loss_bbox_l1 = torch.tensor(0.0, device=pred_logits.device)
      loss_giou = torch.tensor(0.0, device=pred_logits.device)

    losses = {
          'loss_classification': loss_classification,
          'loss_bbox_l1': loss_bbox_l1,
          'loss_giou': loss_giou,
    }

    total_loss = (self.weight_dict['loss_classification'] * loss_classification +
                  self.weight_dict['loss_bbox_l1'] * loss_bbox_l1 +
                  self.weight_dict['loss_giou'] * loss_giou)

    losses['total_loss'] = total_loss
    return losses

# **Config**

In [ ]:
import os 
model_name = "facebook/dinov3-vitb16-pretrain-lvd1689m"

TRAINING_DATA_DIRECTORY = find_existing_path("dataset/train")
VALIDATION_DATA_DIRECTORY = find_existing_path("dataset/val")
SAVE_DIRECTORY = find_existing_path("outputs")
if not all([TRAINING_DATA_DIRECTORY, VALIDATION_DATA_DIRECTORY, SAVE_DIRECTORY]):
    raise FileNotFoundError(
        "Không thể tìm thấy một hoặc nhiều thư mục (dataset/train, dataset/val, outputs). "
        "Hãy đảm bảo rằng cấu trúc thư mục là 'dino_tracker/' chứa notebook, 'dataset', và 'outputs'."
    )
else:
    print("✅ Tất cả đường dẫn đã được xác định thành công.")
os.makedirs(SAVE_DIRECTORY, exist_ok=True)

# Hyperparams
BATCH_SIZE = 4
NUM_WORKERS = 2
TOTAL_EPOCHS = 300
START_EPOCH = 1 # Sẽ được cập nhật nếu tải checkpoint thành công
LEARNING_RATE = 1e-6
WEIGHT_DECAY = 1e-4
GRADIENT_CLIP_NORM = 0.1

# Cấu hình DETR
NUM_CLASSES = 1
NUM_QUERIES = 15
WEIGHT_DICT = {'loss_classification': 1.0, 'loss_bbox_l1': 10.0, 'loss_giou': 5.0}
NO_OBJECT_CLASS_WEIGHT = 0.1

print("✅ Đã hoàn tất cấu hình huấn luyện.")

# **Components**

In [ ]:
import os
import time
import random
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from transformers import AutoImageProcessor, AutoModel

# Backbone vs Processor
print(f"--- Đang tải model backbone '{model_name}' ---")
processor = AutoImageProcessor.from_pretrained(model_name)
backbone_model = AutoModel.from_pretrained(model_name)
backbone_model.to(device)
backbone_model.eval()
for p in backbone_model.parameters():
    p.requires_grad = False

total_params = sum(p.numel() for p in backbone_model.parameters())
trainable_params = sum(p.numel() for p in backbone_model.parameters() if p.requires_grad)
print(f"Backbone loaded. Total params: {total_params:,}. Trainable params: {trainable_params:,}")


print("--- Đang khởi tạo DataLoaders ---")
train_image_paths = find_image_files(TRAINING_DATA_DIRECTORY)
val_image_paths = find_image_files(VALIDATION_DATA_DIRECTORY)

# train
train_augmentations = create_augmentations(is_train=True)
train_dataset = ObjectDetectionDataset(
    image_paths=train_image_paths,
    processor=processor,
    augmentations=train_augmentations
)

# pin_memory chỉ bật khi có CUDA
pin_memory_flag = torch.cuda.is_available()
num_workers_effective = min(NUM_WORKERS, os.cpu_count() or 0)

train_dataloader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=num_workers_effective,
    collate_fn=detection_collate_fn,
    pin_memory=pin_memory_flag
)
print(f"Đã tạo Training DataLoader. Số lượng ảnh: {len(train_dataset)}. Số batch: {len(train_dataloader)}")

# validation
val_augmentations = create_augmentations(is_train=False)
val_dataset = ObjectDetectionDataset(
    image_paths=val_image_paths,
    processor=processor,
    augmentations=val_augmentations
)
val_dataloader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=num_workers_effective,
    collate_fn=detection_collate_fn,
    pin_memory=pin_memory_flag
)
print(f"Đã tạo Validation DataLoader. Số lượng ảnh: {len(val_dataset)}. Số batch: {len(val_dataloader)}")


print("--- Đang khởi tạo Model, Loss, và Optimizer ---")
# model detector
detection_model = DINODetector(
    backbone_model=backbone_model,
    num_object_queries=NUM_QUERIES,
    num_classes=NUM_CLASSES
).to(device)

# Matcher vs Loss
matcher = HungarianMatcher(cost_class=1.0, cost_bbox_l1=5.0, cost_giou=2.0)
loss_function = DETRLoss(
    num_classes=NUM_CLASSES,
    matcher=matcher,
    weight_dict=WEIGHT_DICT,
    no_object_class_weight=NO_OBJECT_CLASS_WEIGHT
).to(device)

# Optimizer (only detector head)
optimizer = optim.AdamW(
    params=detection_model.detection_head.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Scaler cho Mixed Precision Training
use_amp = torch.cuda.is_available()
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


print("--- Đang kiểm tra checkpoint ---")
# Loss history
training_history = {
    'train_loss': [],
    'val_loss': [],
    'learning_rate': [],
    'train_loss_classification': [],
    'train_loss_bbox_l1': [],
    'train_loss_giou': [],
    'val_loss_classification': [],
    'val_loss_bbox_l1': [],
    'val_loss_giou': []
}
best_validation_loss = float("inf")
CHECKPOINT_TO_LOAD = os.path.join(SAVE_DIRECTORY, "checkpoint_epoch_100.pth")

if os.path.exists(CHECKPOINT_TO_LOAD):
    print(f"Đang tải checkpoint từ: {CHECKPOINT_TO_LOAD}")
    checkpoint = torch.load(CHECKPOINT_TO_LOAD, map_location=device)

    # Tải trọng số của detection head
    detection_model.detection_head.load_state_dict(checkpoint['model_state_dict'])

    # Tải trạng thái của optimizer và scaler
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    if checkpoint.get('scaler_state_dict'):
      scaler.load_state_dict(checkpoint['scaler_state_dict'])

    # Cập nhật epoch bắt đầu và best_validation_loss
    START_EPOCH = checkpoint.get('epoch', 0) + 1
    best_validation_loss = checkpoint.get('best_validation_loss', float('inf'))

    if checkpoint.get('training_history'):
      training_history = checkpoint['training_history']

    print(f"Đã tải checkpoint thành công. Tiếp tục training từ epoch {START_EPOCH}.")
    print(f"Validation loss tốt nhất từ lần trước: {best_validation_loss:.4f}")
else:
    print(f"Không tìm thấy checkpoint tại '{CHECKPOINT_TO_LOAD}'. Bắt đầu training từ đầu (epoch 1).")

if device.type == "cuda":
    torch.cuda.empty_cache() # Dọn dẹp cache trước khi đo lường
    start_gpu_memory = round(torch.cuda.max_memory_reserved(device) / 1024**3, 3)
    print(f"✅ Bộ nhớ GPU đã sử dụng sau khi tải model: {start_gpu_memory} GB.")
else:
    start_gpu_memory = 0

# Bắt đầu đếm tổng thời gian huấn luyện
total_training_start_time = time.time()

# **Training**

In [ ]:
print("Start training...")

# Training loop
for epoch in range(START_EPOCH, TOTAL_EPOCHS + 1):
    detection_model.train()
    epoch_train_loss = 0.0
    epoch_train_losses_components = {'loss_classification': 0.0, 'loss_bbox_l1': 0.0, 'loss_giou': 0.0}

    if use_amp: # Chỉ theo dõi VRAM nếu dùng GPU
      torch.cuda.reset_peak_memory_stats(device)

    if len(train_dataloader) == 0:
      print("❌ Training dataloader is empty. Aborting training loop.")
      break

    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch}/{TOTAL_EPOCHS} [Training]")
    for pixel_values_batch, targets_list in progress_bar:
        pixel_values_batch = pixel_values_batch.to(device, non_blocking=True)
        targets_list = [{k: v.to(device, non_blocking=True) if torch.is_tensor(v) else v for k, v in t.items()} for t in targets_list]
        optimizer.zero_grad(set_to_none=True)

        # Foward pass with Automatic Mixed Precision
        with torch.cuda.amp.autocast(enabled=use_amp):
            model_outputs = detection_model(pixel_values_batch)
            loss_dict = loss_function(model_outputs, targets_list)
            total_loss = loss_dict['total_loss']

        if not torch.isfinite(total_loss):
          print(f"WARNING: Đã phát hiện loss không hợp lệ (NaN hoặc Inf). Bỏ qua bước này.")
          continue

        # Backward pass
        scaler.scale(total_loss).backward()

        # Gradient Clipping
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(detection_model.detection_head.parameters(), GRADIENT_CLIP_NORM)

        # Update
        scaler.step(optimizer)
        scaler.update()

        epoch_train_loss += total_loss.item()
        progress_bar.set_postfix(loss=f"{total_loss.item():.4f}")
        for key in epoch_train_losses_components:
          if key in loss_dict:
            epoch_train_losses_components[key] += loss_dict[key].item()

    # scheduler.step()
    average_train_loss = epoch_train_loss / len(train_dataloader)
    avg_cls_loss = epoch_train_losses_components['loss_classification'] / len(train_dataloader) # Loss cho việc phân loại có vật thể hay không, giao động 0-1, giá trị càng nhỏ càng tốt
    avg_l1_loss = epoch_train_losses_components['loss_bbox_l1'] / len(train_dataloader) # Loss cho khoảng cách L1 giữa tọa độ hộp dự đoán và hộp thật
    avg_giou_loss = epoch_train_losses_components['loss_giou'] / len(train_dataloader) # Loss cho độ trùng khớp của hai hộp, giao động 0-2

    # vram dùng sau mỗi epoch
    if use_amp:
      train_vram_peak_bytes = torch.cuda.max_memory_allocated(device)
      train_vram_peak_gb = train_vram_peak_bytes / (1024**3)
    else:
      train_vram_peak_gb = 0.0
    print(f"[Epoch {epoch}] Train Loss: {average_train_loss:.4f} "
          f"(Classification: {avg_cls_loss:.4f}, L1 BBOX: {avg_l1_loss:.4f}, GIoU: {avg_giou_loss:.4f}) | "
          f"LR: {LEARNING_RATE:.6f} | VRAM: {train_vram_peak_gb:.2f} GB")

    training_history['train_loss'].append(average_train_loss)
    # training_history['learning_rate'].append(scheduler.get_last_lr()[0])
    training_history['learning_rate'].append(LEARNING_RATE)
    for key in epoch_train_losses_components:
      training_history[f'train_{key}'].append(epoch_train_losses_components[key] / len(train_dataloader))

    # Validate
    average_val_loss = None
    if val_dataloader is not None:
      detection_model.eval()
      epoch_val_loss = 0.0
      epoch_val_losses_components = {'loss_classification': 0.0, 'loss_bbox_l1': 0.0, 'loss_giou': 0.0}
      with torch.no_grad():
        val_progress_bar = tqdm(val_dataloader, desc=f"Epoch {epoch}/{TOTAL_EPOCHS} [Validation]")
        for pixel_values_batch, targets_list in val_progress_bar:
          pixel_values_batch = pixel_values_batch.to(device, non_blocking=True)
          targets_list = [{k: v.to(device, non_blocking=True) if torch.is_tensor(v) else v for k, v in t.items()} for t in targets_list]

          with torch.cuda.amp.autocast(enabled=use_amp):
            model_outputs = detection_model(pixel_values_batch)
            loss_dict = loss_function(model_outputs, targets_list)
            total_loss = loss_dict['total_loss']

          if torch.isfinite(total_loss):
            epoch_val_loss += total_loss.item()
            for key in epoch_val_losses_components:
                if key in loss_dict:
                    epoch_val_losses_components[key] += loss_dict[key].item()

      average_val_loss = epoch_val_loss / len(val_dataloader)
      print(f"[Epoch {epoch}] Validation Loss: {average_val_loss:.4f}")

      if average_val_loss < best_validation_loss:
        best_validation_loss = average_val_loss
        best_model_path = os.path.join(SAVE_DIRECTORY, "best_detection_head.pth")
        torch.save(detection_model.detection_head.state_dict(), best_model_path)
        print(f"-> Val loss cải thiện. Đã lưu model tốt nhất tại: {best_model_path}")

    training_history['val_loss'].append(average_val_loss)
    for key in epoch_val_losses_components:
      avg_val_component_loss = epoch_val_losses_components[key] / len(val_dataloader)
      history_key = f'val_{key}' # Sẽ tạo ra 'val_loss_classification', etc.
      if history_key not in training_history:
          training_history[history_key] = []
      training_history[history_key].append(avg_val_component_loss)

    # Checkpoint
    checkpoint_path = os.path.join(SAVE_DIRECTORY, f"checkpoint_epoch_{epoch}.pth")
    torch.save({
        'epoch': epoch,
        'model_state_dict': detection_model.detection_head.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'average_train_loss': average_train_loss,
        'average_validation_loss': average_val_loss,
        'best_validation_loss': best_validation_loss,
        'training_history': training_history,
    }, checkpoint_path)
    print(f"Đã lưu checkpoint của Epoch {epoch} tại: {checkpoint_path}\n")
print("Hoàn tất quá trình huấn luyện.")

In [ ]:
total_training_end_time = time.time()
total_runtime_seconds = total_training_end_time - total_training_start_time

print("\n--- Thống kê tài nguyên sau khi hoàn tất huấn luyện ---")
print(f"Tổng thời gian huấn luyện: {total_runtime_seconds:.2f} giây.")
print(f"Tương đương: {total_runtime_seconds / 60:.2f} phút.")

if device.type == "cuda":
    # Lấy đỉnh bộ nhớ của toàn bộ quá trình
    peak_reserved_memory = round(torch.cuda.max_memory_reserved(device) / 1024**3, 3)

    # Tính toán bộ nhớ sử dụng thêm cho quá trình huấn luyện (gradients, activations, etc.)
    used_memory_for_training = round(peak_reserved_memory - start_gpu_memory, 3)

    # Tính toán phần trăm
    peak_percentage = round(peak_reserved_memory / max_memory * 100, 3) if max_memory > 0 else 0
    training_percentage = round(used_memory_for_training / max_memory * 100, 3) if max_memory > 0 else 0

    print(f"\n--- Thống kê VRAM ---")
    print(f"Đỉnh bộ nhớ đã sử dụng (toàn bộ): {peak_reserved_memory} GB.")
    print(f"Đỉnh bộ nhớ sử dụng thêm cho việc huấn luyện: {used_memory_for_training} GB.")
    print(f"Tỷ lệ đỉnh bộ nhớ / dung lượng tối đa: {peak_percentage} %.")
    print(f"Tỷ lệ bộ nhớ huấn luyện / dung lượng tối đa: {training_percentage} %.")
else:
    print("\nKhông sử dụng GPU, không có thống kê VRAM.")

# **Chart**

In [ ]:
import matplotlib.pyplot as plt
import os

def plot_training_analysis(history, save_directory, filename_prefix="training_analysis"):
    def setup_ax(ax, title, xlabel="Epoch", ylabel="Loss"):
        ax.set_title(title, fontsize=12)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.grid(True, linestyle='--', alpha=0.6)
        if ax.get_legend_handles_labels()[1]:
            ax.legend()

    epochs = range(1, len(history['train_loss']) + 1)
    fig, axs = plt.subplots(2, 2, figsize=(18, 12))
    fig.suptitle('Phân Tích Chi Tiết Quá Trình Huấn Luyện', fontsize=18)

    # Biểu đồ 1: Training vs. Validation Loss tổng thể
    ax = axs[0, 0]
    ax.plot(epochs, history['train_loss'], 'o-', color='royalblue', label='Training Loss')
    if history.get('val_loss'): # Kiểm tra key 'val_loss' có tồn tại và không rỗng
        ax.plot(epochs, history['val_loss'], 'o-', color='orangered', label='Validation Loss')
    setup_ax(ax, title='Training & Validation Loss')

    # Biểu đồ 2: Learning Rate
    ax = axs[0, 1]
    if history.get('learning_rate'):
        ax.plot(epochs, history['learning_rate'], 's-', color='green', label='Learning Rate')
    setup_ax(ax, title='Learning Rate Schedule', ylabel='Learning Rate')

    # Biểu đồ 3: Thành phần Loss - Phân loại (Classification)
    ax = axs[1, 0]
    if history.get('train_loss_classification'):
        ax.plot(epochs, history['train_loss_classification'], '^-', color='darkcyan', label='Train Classification Loss')
    if history.get('val_loss_classification'):
        ax.plot(epochs, history['val_loss_classification'], '^-', color='purple', label='Val Classification Loss')
    setup_ax(ax, title='Thành Phần Loss: Phân Loại')

    # Biểu đồ 4: Thành phần Loss - Bounding Box
    ax = axs[1, 1]
    if history.get('train_loss_bbox_l1'):
        ax.plot(epochs, history['train_loss_bbox_l1'], 'D--', color='darkblue', label='Train BBox L1')
    if history.get('val_loss_bbox_l1'):
        ax.plot(epochs, history['val_loss_bbox_l1'], 'D--', color='crimson', label='Val BBox L1')
    if history.get('train_loss_giou'):
        ax.plot(epochs, history['train_loss_giou'], 'p:', color='darkgoldenrod', label='Train GIoU')
    if history.get('val_loss_giou'):
        ax.plot(epochs, history['val_loss_giou'], 'p:', color='saddlebrown', label='Val GIoU')
    setup_ax(ax, title='Thành Phần Loss: Bounding Box')

    plt.tight_layout(rect=[0, 0.03, 1, 0.96])

    version = 1
    figure_path = os.path.join(save_directory, f"{filename_prefix}_{version}.png")
    while os.path.exists(figure_path):
        version += 1
        figure_path = os.path.join(save_directory, f"{filename_prefix}_{version}.png")

    try:
        fig.savefig(figure_path)
        print(f"✅ Đã lưu biểu đồ phân tích tại: {figure_path}")
        plt.show()
    except Exception as e:
        print(f"❌ Lỗi khi lưu biểu đồ: {e}")

plot_training_analysis(training_history, SAVE_DIRECTORY)